# 02 — CBraMod PhysioNet-MI: authors' fine-tuning code

**Run your existing `00_colab_setup(2).ipynb` first**, in this Colab runtime. Leave it unchanged.

This notebook downloads the **official CBraMod repository** into a separate `/content/CBraMod` directory and executes the authors' actual `preprocessing/preprocessing_physio.py` and `finetune_main.py`. It does **not** reimplement the EEG preprocessing, model, optimizer, or evaluator.

Only Colab adaptations: input/output paths for the preprocessing script; extraction of the six EDF runs from your existing ZIP; and a minimal fix for repeated LMDB openings in the official data loader. **No mixed precision or custom speed/training changes.**

## 1. Paths (match your setup notebook)

In [ ]:
from pathlib import Path
import subprocess, sys, re, shutil, pickle, csv
from zipfile import ZipFile
from pathlib import PurePosixPath

# From your existing setup notebook; keep it as-is.
REPO_DIR = Path('/content/cbramod-head-interp')  # your GitHub checkout (not modified here)
DATA_DIR = Path('/content/data')                  # Colab temporary disk
ZIP_PATH = Path('/content/drive/MyDrive/CBraMod-MI/eeg-motor-movementimagery-dataset-1.0.0.zip')

# Only the authors' code gets its own separate folder.
AUTHORS_REPO = Path('/content/CBraMod')
WORK_DIR = DATA_DIR / 'physionet_mi'
RAW_DIR = WORK_DIR / 'files'
LMDB_DIR = WORK_DIR / 'processed_average'
CHECKPOINT_DIR = WORK_DIR / 'pretrained_weights'
RUN_DIR = WORK_DIR / 'runs'
DRIVE_RESULTS = ZIP_PATH.parent / 'results' / 'physionet_mi_4class'

WORK_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.is_file(), f'Dataset not found: {ZIP_PATH}'
print('Dataset:', ZIP_PATH)
print('Temporary working data:', WORK_DIR)
print('Drive results:', DRIVE_RESULTS)

## 2. Download the original authors' implementation

In [ ]:
# This is NOT your GitHub project. The separate official repository supplies
# the authors' preprocessing, model, trainer and evaluator.
OFFICIAL_COMMIT = 'b9e961003214326972c567eff390e75b0287e32a'
if not (AUTHORS_REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/wjq-learning/CBraMod.git',
                    str(AUTHORS_REPO)], check=True)
subprocess.run(['git', '-C', str(AUTHORS_REPO), 'checkout', '--quiet', OFFICIAL_COMMIT], check=True)

# The upstream requirements do not pin versions. Use installed Colab PyTorch/GPU drivers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(AUTHORS_REPO / 'requirements.txt'), 'huggingface_hub'], check=True)
import lmdb, torch
from huggingface_hub import hf_hub_download
print('Upstream commit:', OFFICIAL_COMMIT)
print('GPU:', torch.cuda.get_device_name(0))

## 3. Extract the original imagery EDF files

The authors' script expects `files/S001/S001R04.edf`, etc. We arrange the ZIP **in that exact layout** under `/content/data/physionet_mi/files/`. Only runs 04, 06, 08, 10, 12 and 14 are needed.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(ZIP_PATH) as archive:
    edfs = []
    for info in archive.infolist():
        name = PurePosixPath(info.filename).name
        if re.fullmatch(r'S\d{3}R(04|06|08|10|12|14)\.edf', name, flags=re.I):
            filename = name[:7].upper() + '.edf'
            edfs.append((info, filename))
    assert len(edfs) == 654, f'Expected 654 imagery recordings; found {len(edfs)}'

    for info, filename in edfs:
        target = RAW_DIR / filename[:4] / filename
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.stat().st_size == info.file_size:
            continue
        with archive.open(info) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

print('Imagery recordings ready:', len(edfs))
print('Example:', RAW_DIR / 'S001' / 'S001R04.edf')

## 4. Run **the authors'** PhysioNet preprocessing script

Instead of rewriting their EEG operations, copy `preprocessing_physio.py` and replace **only its two hard-coded disk paths**. Its filtering, referencing, epoching, channel order, labels, subject split, and LMDB writing remain the original code. Full preprocessing can take a while on Colab CPU.

In [ ]:
official_preprocess = AUTHORS_REPO / 'preprocessing' / 'preprocessing_physio.py'
colab_preprocess = WORK_DIR / 'preprocessing_physio_colab_paths.py'
source = official_preprocess.read_text()

old_root = "root_dir = '/data/datasets/eeg-motor-movementimagery-dataset-1.0.0/files'"
old_db = "db = lmdb.open('/data/datasets/eeg-motor-movementimagery-dataset-1.0.0/processed_average', map_size=4614542346)"
assert source.count(old_root) == 1 and source.count(old_db) == 1, 'Upstream preprocessing changed; review its paths.'
source = source.replace(old_root, f'root_dir = {str(RAW_DIR)!r}')
source = source.replace(old_db, f'db = lmdb.open({str(LMDB_DIR)!r}, map_size=4614542346)')
compile(source, str(colab_preprocess), 'exec')
colab_preprocess.write_text(source)
print('Authors’ script with ONLY paths adapted:', colab_preprocess)

In [ ]:
# Do not preprocess twice if this Colab runtime already has a completed LMDB.
completed_db = False
if (LMDB_DIR / 'data.mdb').exists():
    try:
        db = lmdb.open(str(LMDB_DIR), readonly=True, lock=False, readahead=False)
        with db.begin() as tx:
            completed_db = tx.get(b'__keys__') is not None
        db.close()
    except lmdb.Error:
        completed_db = False

if completed_db:
    print('Processed LMDB already exists; using it.')
else:
    # Execute the actual upstream preprocessing (path-adjusted copy).
    preprocess_log = WORK_DIR / 'preprocessing_log.txt'
    with preprocess_log.open('w') as log:
        subprocess.run([sys.executable, '-u', str(colab_preprocess)],
                       cwd=str(AUTHORS_REPO), stdout=log, stderr=subprocess.STDOUT, check=True)
    print('Authors’ preprocessing finished. Log:', preprocess_log)

In [ ]:
# Small sanity check; the authors' published sample count is 9,837 trials.
db = lmdb.open(str(LMDB_DIR), readonly=True, lock=False, readahead=False)
with db.begin() as tx:
    keys = pickle.loads(tx.get(b'__keys__'))
    sample = pickle.loads(tx.get(keys['train'][0].encode()))
db.close()
print('Train / validation / test:', [len(keys[k]) for k in ('train', 'val', 'test')])
print('Total trials:', sum(len(v) for v in keys.values()))
print('Trial shape:', sample['sample'].shape, '| label:', sample['label'])
assert sum(len(v) for v in keys.values()) == 9837, 'Unexpected trial count: compare preprocessing and dataset version.'

## 5. Original model and pretrained checkpoint

Use the weights published by the CBraMod authors. Their `physio_dataset.py` has an LMDB opening conflict when train/validation/test datasets are created in one process on this setup; the one connection-sharing fix below changes no samples or training logic.

In [ ]:
PRETRAINED = Path(hf_hub_download(repo_id='weighting666/CBraMod',
                                   filename='pretrained_weights.pth',
                                   local_dir=str(CHECKPOINT_DIR)))

loader_file = AUTHORS_REPO / 'datasets' / 'physio_dataset.py'
original_open = 'self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)'
shared_open = ("if not hasattr(CustomDataset, '_shared_db'):\n"
               "            CustomDataset._shared_db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)\n"
               "        self.db = CustomDataset._shared_db")
loader = loader_file.read_text()
if original_open in loader:
    patched = loader.replace(original_open, shared_open)
    compile(patched, str(loader_file), 'exec')
    loader_file.write_text(patched)
elif shared_open not in loader:
    raise RuntimeError('Official LMDB loader has changed; review it before fine-tuning.')

print('Authors’ pretrained checkpoint:', PRETRAINED)

## 6. Fine-tune using the authors' `finetune_main.py`

The paper reports mean ± standard deviation over five runs. **The authors' exact five seeds are not identified in the published code**, so this defaults to their script's seed **3407** for a faithful *single run*. For a five-run estimate, add four seeds to `SEEDS`—but do not call that the exact published five-seed replication.

All training defaults come from the original `finetune_main.py` (50 epochs, batch size 64, AdamW, cosine scheduler, label smoothing 0.1, multi-LR, trainable backbone). We change only dataset, model and checkpoint paths, dataset name, number of classes, and CUDA index.

In [ ]:
SEEDS = [3407]  # The authors' fine-tune script's default seed.
RUN_DIR.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    seed_dir = RUN_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-u', 'finetune_main.py',
        '--downstream_dataset', 'PhysioNet-MI',
        '--datasets_dir', str(LMDB_DIR),
        '--num_of_classes', '4',
        '--foundation_dir', str(PRETRAINED),
        '--model_dir', str(seed_dir),
        '--cuda', '0',
        '--seed', str(seed),
    ]
    print('Fine-tuning with original authors’ trainer; seed:', seed, flush=True)
    log_path = seed_dir / 'training_log.txt'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=str(AUTHORS_REPO),
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        status = process.wait()
    if status != 0:
        raise RuntimeError(f'Fine-tuning failed. Inspect {log_path}')

## 7. Keep the authors' model weights and test results on Drive

In [ ]:
rows = []
pattern = re.compile(r'Test Evaluation: acc: ([0-9.]+), kappa: ([0-9.]+), f1: ([0-9.]+)')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
for seed in SEEDS:
    seed_dir = RUN_DIR / f'seed_{seed}'
    log_text = (seed_dir / 'training_log.txt').read_text()
    found = pattern.findall(log_text)
    if not found:
        raise RuntimeError(f'No test evaluation in training log for seed {seed}')
    acc, kappa, f1 = map(float, found[-1])
    rows.append({'seed': seed, 'balanced_accuracy': acc, 'kappa': kappa, 'weighted_f1': f1})
    shutil.copytree(seed_dir, DRIVE_RESULTS / f'seed_{seed}', dirs_exist_ok=True)

with (DRIVE_RESULTS / 'test_scores.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['seed', 'balanced_accuracy', 'kappa', 'weighted_f1'])
    writer.writeheader()
    writer.writerows(rows)
(DRIVE_RESULTS / 'authors_commit.txt').write_text(OFFICIAL_COMMIT + '\n')

for row in rows:
    print(row)
print('Saved to:', DRIVE_RESULTS)

### What was changed (and what wasn't)

- **Unchanged authors' code:** model, optimizer, scheduler, loss, evaluator, all preprocessing operations and event-label logic.
- **Path adjustments:** authors' preprocessing accepts the EDF hierarchy under `/content/data/physionet_mi/files/` and creates LMDB under `/content/data/physionet_mi/processed_average/`.
- **Technical compatibility fix:** one shared read-only LMDB connection in the official dataset class. No data or hyperparameter changes.
- **Output:** `/content/drive/MyDrive/CBraMod-MI/results/physionet_mi_4class/seed_3407/` (model checkpoint and log) plus `test_scores.csv`.

**Sources:** [official implementation](https://github.com/wjq-learning/CBraMod) · [preprocessing_physio.py](https://github.com/wjq-learning/CBraMod/blob/b9e961003214326972c567eff390e75b0287e32a/preprocessing/preprocessing_physio.py) · [finetune_main.py](https://github.com/wjq-learning/CBraMod/blob/b9e961003214326972c567eff390e75b0287e32a/finetune_main.py).